In [0]:
from config import BRONZE_SCHEMA, SILVER_SCHEMA
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

In [0]:
pesquisa_satisfacao_bronze = spark.table(f'{BRONZE_SCHEMA}.pesquisa_satisfacao')
# display(pesquisa_satisfacao_bronze)

In [0]:
pesquisa_sem_duplicados = pesquisa_satisfacao_bronze.drop('ingestion_datetime').dropDuplicates(['id_pesquisa'])
_, passou = dq_check_unique('pesquisa_sem_duplicados', 'id_pesquisa único', pesquisa_sem_duplicados, ['id_pesquisa'])
if not passou:
    raise ValueError('Existem registros diferentes para o mesmo id_pesquisa')
_, passou = dq_check('pesquisa_sem_duplicados', 'id_pesquisa não nulo', pesquisa_sem_duplicados, F.col('id_pesquisa').isNotNull())
if not passou:
    raise ValueError('Existem pesquisas sem id_pesquisa')

In [0]:
# Converte a coluna de nota para inteiro
pesquisa_silver = (
    pesquisa_sem_duplicados
    .withColumn('nota_atendimento', F.col('nota_atendimento').try_cast('int'))
    .withColumn('nota_atendimento',
        F.when((F.col('nota_atendimento') >= 1) & (F.col('nota_atendimento') <= 5), F.col('nota_atendimento'))
        .otherwise(F.lit(None))
    )
)
# display(pesquisa_silver)

In [0]:
pesquisa_silver.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_SCHEMA}.pesquisa_satisfacao')